# Scrape J! Archive Episodes by Season

Download episodes from [j-archive.com](https://j-archive.com) and save them to a local DuckDB database.

This notebook scrapes **Season 1** (first 10 episodes only, as a test).
The site's `robots.txt` is checked before any requests, and the `Crawl-delay` is respected.

In [ ]:
import logging

from tqdm.notebook import tqdm

from jt3 import save_episode
from jt3.crawler import check_robots, episode_url, fetch_episode, get_season_game_ids

logging.basicConfig(level=logging.INFO)

## Check robots.txt

Verify that j-archive.com allows us to crawl, and read the required delay between requests.

In [ ]:
crawl_delay = check_robots()

if crawl_delay is None:
    raise PermissionError("robots.txt disallows crawling — aborting.")

print(f"Crawling allowed. Crawl-delay: {crawl_delay}s")

## Fetch Season 1 game IDs

Get all episode game IDs for Season 1 from the season page, then take the first 10 (chronological order) as a test.

In [ ]:
import time

import requests

# Season page lists newest-first; reverse for chronological order
all_game_ids = get_season_game_ids(1)
all_game_ids.reverse()

# Take first 10 as a test
game_ids = all_game_ids[:10]
print(f"Season 1: {len(all_game_ids)} episodes total, fetching first {len(game_ids)}")

episodes = []

for i, gid in enumerate(pbar := tqdm(game_ids, desc="Fetching episodes")):
    if i > 0 and crawl_delay > 0:
        time.sleep(crawl_delay)

    url = episode_url(gid)
    pbar.set_postfix(game_id=gid)

    try:
        ep = fetch_episode(url)
    except requests.HTTPError as exc:
        print(f"  Skipping game_id={gid}: {exc}")
        continue

    save_episode(ep)
    episodes.append(ep)

print(f"\nSaved {len(episodes)} episodes to DuckDB.")

## Summary

List all episodes that were saved.

In [ ]:
for ep in episodes:
    print(
        f"game_id={ep.game_id:>5}  "
        f"show=#{ep.show_number or '???':>5}  "
        f"aired={ep.air_date or 'unknown'}  "
        f"clues={sum(len(cat.clues) for rnd in ep.rounds for cat in rnd.categories)}"
    )